# Benchmark v2 Kaggle runner

This notebook runs the new `evaluation.benchmark_v2` pipeline on the curated subset from `included_samples.json`.

It defaults to a single-split `dev` smoke run. Do not treat `dev` results as final benchmark numbers.

In [ ]:
# Clone the repository
!git clone https://github.com/buinguyenkhai/stock-report-agent-20251.git
%cd stock-report-agent-20251

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-vie libtesseract-dev libleptonica-dev
%pip install -q pymupdf
%pip install -q -r requirements.txt
%pip install -q "docling[tesserocr]"
%pip install -q marker-pdf surya-ocr

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"

DATASET_ROOT = "data/benchmark_v2"
PREDICTIONS_ROOT = "results/benchmark_v2_hybrid_included"
RESULT_JSON = "results/benchmark_v2_hybrid_included_dev.json"
DEBUG_DIFF_JSON = "results/benchmark_v2_debug_diffs.json"
ENGINE = "hybrid"
ENGINE_NAME = "hybrid_docling"
SPLIT = "dev"
INCLUDE_SCOPE = "included"
DEVICE = "cuda"
HYBRID_THRESHOLD = 0.90
HYBRID_NUMBER_THRESHOLD = 0.95
ENABLE_STRUCTURED = bool(os.getenv("OPENROUTER_API_KEY"))

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"Structured extraction enabled: {ENABLE_STRUCTURED}")
if not ENABLE_STRUCTURED:
    print("OPENROUTER_API_KEY is missing. Notebook will run raw OCR benchmarking only.")

In [ ]:
# Validate the curated subset before spending GPU time.
dataset_root = Path(DATASET_ROOT)
manifest = json.loads((dataset_root / "manifest.json").read_text(encoding="utf-8"))
include_registry = json.loads((dataset_root / "included_samples.json").read_text(encoding="utf-8"))
included_ids = include_registry.get("included_sample_ids", [])
sample_map = {row["sample_id"]: row for row in manifest.get("samples", [])}

missing_from_manifest = [sid for sid in included_ids if sid not in sample_map]
missing_artifacts = []
for sid in included_ids:
    row = sample_map.get(sid)
    if row is None:
        continue
    required = [
        row["gt_markdown_path"],
        row["gt_structured_path"],
        row["page_image_path"],
        f"gt_csv/{sid}/cells.csv",
        f"gt_csv/{sid}/rows.csv",
    ]
    if row.get("gt_table_cells_path"):
        required.append(row["gt_table_cells_path"])
    for rel in required:
        if not (dataset_root / rel).exists():
            missing_artifacts.append((sid, rel))

print(f"Manifest samples: {len(manifest.get('samples', []))}")
print(f"Included samples: {len(included_ids)}")
print(f"Included companies: {sorted({sid.split('_')[0] for sid in included_ids})}")
splits = sorted({sample_map[sid]['split'] for sid in included_ids if sid in sample_map})
print(f"Included splits: {splits}")

if missing_from_manifest:
    print("Missing from manifest:")
    for sid in missing_from_manifest:
        print(" -", sid)

if missing_artifacts:
    print("Missing required artifacts:")
    for sid, rel in missing_artifacts:
        print(f" - {sid}: {rel}")

if not included_ids:
    raise SystemExit("included_samples.json is empty")
if missing_from_manifest or missing_artifacts:
    raise SystemExit("Dataset validation failed; fix included samples/artifacts before running benchmark")

print("Dataset validation passed")

In [ ]:
predict_cmd = [
    sys.executable,
    "-m",
    "evaluation.benchmark_v2.predict",
    "--dataset-root", DATASET_ROOT,
    "--output-root", PREDICTIONS_ROOT,
    "--engine", ENGINE,
    "--split", SPLIT,
    "--include-scope", INCLUDE_SCOPE,
    "--device", DEVICE,
    "--hybrid-threshold", str(HYBRID_THRESHOLD),
    "--hybrid-number-threshold", str(HYBRID_NUMBER_THRESHOLD),
]
if not ENABLE_STRUCTURED:
    predict_cmd.append("--raw-only")

print("Running:", " ".join(predict_cmd))
subprocess.run(predict_cmd, check=True)

In [ ]:
run_cmd = [
    sys.executable,
    "-m",
    "evaluation.benchmark_v2.run",
    "--dataset-root", DATASET_ROOT,
    "--predictions-root", PREDICTIONS_ROOT,
    "--engine-name", ENGINE_NAME,
    "--split", SPLIT,
    "--include-scope", INCLUDE_SCOPE,
    "--output", RESULT_JSON,
]
print("Running:", " ".join(run_cmd))
subprocess.run(run_cmd, check=True)

result = json.loads(Path(RESULT_JSON).read_text(encoding="utf-8"))
print(json.dumps(result["summary"], ensure_ascii=False, indent=2))

In [ ]:
diff_cmd = [
    sys.executable,
    "-m",
    "evaluation.benchmark_v2.debug_diffs",
    "--dataset-root", DATASET_ROOT,
    "--predictions-root", PREDICTIONS_ROOT,
    "--split", SPLIT,
    "--include-scope", INCLUDE_SCOPE,
    "--output", DEBUG_DIFF_JSON,
]
print("Running:", " ".join(diff_cmd))
subprocess.run(diff_cmd, check=True)
print(f"Saved debug diffs to {DEBUG_DIFF_JSON}")

In [ ]:
!zip -r results.zip results